# Bronze → Silver Layer Migration

This notebook performs the data exploration and migration from the Bronze layer (`01-bronze`) to the Silver layer (`02-silver`). 

The Silver layer will contain cleaned, typed, and deduplicated tables.

Silver Design strategy approache: Kinball approach  
- Botton-up: Business process driven instead of enterprise
- Do not apply 3 Normal Form and do not consider SCD7 (slowly changing dimension type 7): 
    - Speed vs Data consistency with high integrity
    - High data redundancy vs Highly normalized data applying 3NF
    - Decision: Keep PK as hash-based value


## General Bronze Data Exploration Analysis

| `geolocation` | 1,000,163 | 280K duplicate rows, lat/lng as STRING |
| `order_items` | 112,650 | shipping_limit_date as STRING |
| `order_payments` | 103,886 | Clean — minor standardization needed |
| `order_reviews` | 104,162 | review_score as STRING with invalid values (dates, Portuguese text), 1,205 duplicate review_ids |
| `orders` | 99,441 | All date columns stored as STRING |
| `product_category_name_translation` | 71 | Clean |
| `products` | 32,951 | 610 null categories, column name typos (`lenght`) |
| `sellers` | 3,095 | Clean |

## Some general transformations for this layer decisions:

- **Column pruning**: Dropping all null columns (_rescued_data) and bronze metadata columns (source_file, ingestion_time)
- **Type casting**: Convert datetime string columns into TIMESTAMP type, lat/lng strings → `DOUBLE`, review_score string → `INT`
- **Data cleaning**: Filter invalid review_scores (keep 1–5), fix column name typos (`lenght` → `length`)
- **Deduplication**: Remove duplicate `review_id` and geolocation rows
- **Standardization**: TRIM text fields, UPPER state codes
- **Metadata**: Add `record_ingestion_timestamp` column to all Silver tables



In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;
SET TIME ZONE 'America/Sao_Paulo';

## Data Exploration Analysis 
This session will explore the bronze ingested data to understand how the data has been ingested and assess the migration to silver design

In [0]:
%sql

-- Explore bronze customers: schema overview + duplicate customer_id check

-- Quick look at the table structure and sample rows
DESCRIBE `puc_data_specialist_de_2026_09`.`01-bronze`.customers;

SELECT * FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers LIMIT 10;

-- Identify which customer_ids are duplicated (if any) and how many times
SELECT
  customer_unique_id,
  COUNT(*) AS occurrence_count
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers
WHERE customer_unique_id IS NOT NULL
GROUP BY customer_unique_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

-- Checking some of the duplicated customer_ids
SELECt c.customer_unique_id, c.customer_id
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers c
WHERE customer_unique_id = '8d50f5eadf50201ccdcedfb9e2ac8455';

-- Checking the relation with order table to see if there are multiple orders for the same customer_id
SELECT o.order_id, o.customer_id, c.customer_unique_id
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.orders o, `puc_data_specialist_de_2026_09`.`01-bronze`.customers c
WHERE o.customer_id = c.customer_id AND c.customer_unique_id = 'd44ccec15f5f86d14d6a2cfa67da1975';

-- Checking what has changed on customer data for the customer that has changed its customer_id
WITH customer_duplicated AS (
    SELECT
        customer_unique_id,
        COUNT(*) AS occurrence_count
    FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers
    WHERE customer_unique_id IS NOT NULL
    GROUP BY customer_unique_id
    HAVING COUNT(*) > 1
),  customer_data_grouped AS (  
    SELECT
    c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix
    FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers c
    JOIN customer_duplicated cd
    ON c.customer_unique_id = cd.customer_unique_id
    GROUP BY c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix
), customer_disticted_data AS (
    SELECT 
        cg.customer_unique_id, count(*) occurrence_count
    FROM customer_data_grouped cg
    GROUP BY cg.customer_unique_id
    HAVING occurrence_count > 1
    ORDER BY occurrence_count DESC
)
SELECT  c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix, cdd.occurrence_count
FROM customer_disticted_data cdd, customer_data_grouped c
WHERE cdd.customer_unique_id = c.customer_unique_id
ORDER BY cdd.occurrence_count, cdd.customer_unique_id  DESC;

-- Checking the results of the above query in customer table direclty.  
SELECT DISTINCT c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers c 
WHERE c.customer_unique_id in ('3e43e6105506432c953e165fb2acf44c', 'd44ccec15f5f86d14d6a2cfa67da1975')
ORDER BY c.customer_unique_id;





## Customer Data Analysis

### Considerations
- It has been identified that the original data uses a hash value as a key (cutomer_unique_id and customer_id), must probably generated by a MD5 algorithm. 
    - When migrating to Silver layer, it has been considered if a creation of a INT/BIGINT surrogate key is a good approach
    - A reason to consider changing the key's to int is performance on joins. Joins using string have a worse performance when comparing with numeric id's
    - For the pourpouse of this project, the ammount of data is not significative to impact performance
    - Spring joins does not have a significant worse performance in Apache Spark, as discussed on [this topic](https://discord.com/channels/1521498663011221615/1521498664608989200/1552793656484696125). 
    - Therefore, after a deep research, and using [this discution](https://community.databricks.com/t5/warehousing-analytics/guid-or-concatenated-string-as-a-primary-key-in-silver-and-gold/m-p/150204/highlight/true#M2524) as reference, in order to garantee indepotency when recreating Silver layer, it has been decided to keep to keep the hash id comming from bronze layer
- customer duplicated ids (cutomer_unique_id and customer_id)
    - After running the above SQL's its possible to check that 
!['customer duplicated data'](customer_id_duplicated_data_reason.png)

In [0]:
%sql

-- Table customers — select native columns, trim text, standardize state. 
-- Adding a _record_created_timestamp column for auditing purposes
-- Customer City, Zipcode and State column will not be normalized using 3NF as per Inmon approach
-- Keeping the original Primary key (customer_id) as hash based value

CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.customers AS
SELECT
  TRIM(customer_id) AS customer_id,
  NULLIF(TRIM(customer_unique_id), '') AS customer_unique_id,
  cast(NULLIF(TRIM(customer_zip_code_prefix), '') AS int)AS customer_zip_code_prefix,
  NULLIF(TRIM(customer_city), '') AS customer_city,
  NULLIF(UPPER(TRIM(customer_state)), '') AS customer_state,
  current_timestamp() AS record_ingestion_timestamp
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers
WHERE customer_id IS NOT NULL;

In [0]:
%sql
-- Silver: geolocation — deduplicate, cast lat/lng to DOUBLE, trim text
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.geolocation AS
SELECT
  TRIM(geolocation_zip_code_prefix) AS geolocation_zip_code_prefix,
  CAST(geolocation_lat AS DOUBLE) AS geolocation_lat,
  CAST(geolocation_lng AS DOUBLE) AS geolocation_lng,
  TRIM(geolocation_city) AS geolocation_city,
  UPPER(TRIM(geolocation_state)) AS geolocation_state,
  current_timestamp() AS silver_ingestion_time
FROM (
  SELECT
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state,
    ROW_NUMBER() OVER (
      PARTITION BY geolocation_zip_code_prefix, geolocation_lat, geolocation_lng
      ORDER BY geolocation_city
    ) AS rn
  FROM `puc_data_specialist_de_2026_09`.`01-bronze`.geolocation
  WHERE geolocation_zip_code_prefix IS NOT NULL
)
WHERE rn = 1;

In [0]:
%sql
-- Silver: orders — cast all date strings to TIMESTAMP, keep only native columns
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.orders AS
SELECT
  TRIM(order_id) AS order_id,
  TRIM(customer_id) AS customer_id,
  TRIM(order_status) AS order_status,
  TRY_CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
  TRY_CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
  TRY_CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
  TRY_CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
  TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date,
  current_timestamp() AS silver_ingestion_time
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.orders
WHERE order_id IS NOT NULL;

In [0]:
%sql
-- Silver: order_items — cast shipping_limit_date, keep native columns
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.order_items AS
SELECT
  TRIM(order_id) AS order_id,
  order_item_id,
  TRIM(product_id) AS product_id,
  TRIM(seller_id) AS seller_id,
  TRY_CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
  price,
  freight_value,
  current_timestamp() AS silver_ingestion_time
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.order_items
WHERE order_id IS NOT NULL;

In [0]:
%sql
-- Silver: order_payments — standardize payment_type, keep native columns
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.order_payments AS
SELECT
  TRIM(order_id) AS order_id,
  payment_sequential,
  LOWER(TRIM(payment_type)) AS payment_type,
  payment_installments,
  payment_value,
  current_timestamp() AS silver_ingestion_time
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.order_payments
WHERE order_id IS NOT NULL;

In [0]:
%sql
-- Silver: order_reviews — cast review_score to INT (filter invalid), deduplicate, cast dates to TIMESTAMP
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.order_reviews AS
SELECT
  TRIM(review_id) AS review_id,
  TRIM(order_id) AS order_id,
  CAST(review_score AS INT) AS review_score,
  TRIM(review_comment_title) AS review_comment_title,
  TRIM(review_comment_message) AS review_comment_message,
  TRY_CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
  TRY_CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
  current_timestamp() AS silver_ingestion_time
FROM (
  SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp,
    ROW_NUMBER() OVER (
      PARTITION BY review_id
      ORDER BY review_creation_date DESC
    ) AS rn
  FROM `puc_data_specialist_de_2026_09`.`01-bronze`.order_reviews
  WHERE review_id IS NOT NULL
    AND review_score IN ('1', '2', '3', '4', '5')
)
WHERE rn = 1;

In [0]:
%sql
-- Silver: products — fix column name typos (lenght → length), handle null categories
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.products AS
SELECT
  TRIM(product_id) AS product_id,
  COALESCE(TRIM(product_category_name), 'unknown') AS product_category_name,
  product_name_lenght AS product_name_length,
  product_description_lenght AS product_description_length,
  product_photos_qty,
  product_weight_g,
  product_length_cm,
  product_height_cm,
  product_width_cm,
  current_timestamp() AS silver_ingestion_time
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.products
WHERE product_id IS NOT NULL;

In [0]:
%sql
-- Silver: product_category_name_translation — simple passthrough with trimming
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.product_category_name_translation AS
SELECT
  TRIM(product_category_name) AS product_category_name,
  TRIM(product_category_name_english) AS product_category_name_english,
  current_timestamp() AS silver_ingestion_time
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.product_category_name_translation
WHERE product_category_name IS NOT NULL;

In [0]:
%sql
-- Silver: sellers — trim text, standardize state
CREATE OR REPLACE TABLE `puc_data_specialist_de_2026_09`.`02-silver`.sellers AS
SELECT
  TRIM(seller_id) AS seller_id,
  TRIM(seller_zip_code_prefix) AS seller_zip_code_prefix,
  TRIM(seller_city) AS seller_city,
  UPPER(TRIM(seller_state)) AS seller_state,
  current_timestamp() AS silver_ingestion_time
FROM `puc_data_specialist_de_2026_09`.`01-bronze`.sellers
WHERE seller_id IS NOT NULL;

In [0]:
%sql
-- Validation: Compare row counts Bronze vs Silver
SELECT 'customers' AS table_name,
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.customers) AS bronze_count,
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.customers) AS silver_count
UNION ALL SELECT 'geolocation',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.geolocation),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.geolocation)
UNION ALL SELECT 'order_items',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.order_items),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.order_items)
UNION ALL SELECT 'order_payments',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.order_payments),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.order_payments)
UNION ALL SELECT 'order_reviews',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.order_reviews),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.order_reviews)
UNION ALL SELECT 'orders',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.orders),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.orders)
UNION ALL SELECT 'products',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.products),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.products)
UNION ALL SELECT 'product_category_name_translation',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.product_category_name_translation),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.product_category_name_translation)
UNION ALL SELECT 'sellers',
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`01-bronze`.sellers),
  (SELECT COUNT(*) FROM `puc_data_specialist_de_2026_09`.`02-silver`.sellers)
ORDER BY table_name;

In [0]:
%sql
-- Validation: Check Silver data quality — nulls and type correctness
SELECT 'order_reviews.review_score' AS check_target,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS null_count,
  COUNT(DISTINCT review_score) AS distinct_values
FROM `puc_data_specialist_de_2026_09`.`02-silver`.order_reviews
UNION ALL SELECT 'orders.order_purchase_timestamp',
  COUNT(*),
  SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END),
  0
FROM `puc_data_specialist_de_2026_09`.`02-silver`.orders
UNION ALL SELECT 'products.product_category_name',
  COUNT(*),
  SUM(CASE WHEN product_category_name = 'unknown' THEN 1 ELSE 0 END),
  COUNT(DISTINCT product_category_name)
FROM `puc_data_specialist_de_2026_09`.`02-silver`.products
UNION ALL SELECT 'geolocation.geolocation_lat (type check)',
  COUNT(*),
  SUM(CASE WHEN geolocation_lat IS NULL THEN 1 ELSE 0 END),
  0
FROM `puc_data_specialist_de_2026_09`.`02-silver`.geolocation;